# Demo

## Import data and packages
The majority of code used in this notebook comes from functions in a local package. Take a look at the Python files in `uoa_lecture`. 

In [ ]:
from uoa_lecture import *
import geopandas as gpd
import pandas as pd
import momepy
import libpysal
import matplotlib.pyplot as plt

In [ ]:
buildings_raw = gpd.read_parquet("../data/buidings_osm_Auckland Central.parquet")
streets_raw = gpd.read_parquet("../data/streets_osm_Auckland Central.parquet")

## Generate morphometrics
Here we're creating a small suite of morphometrics as if we didn't have building height information. Most urban areas in New Zealand don't have this feature available in locally produced datasets. 

In [ ]:
buildings, \
streets, \
tessellation, \
blocks, \
merged, \
percentiles_joined = generate_morphometrics(buildings_raw, streets_raw)

## Classify Auckland CBD
Let's run a clustering algorithm using all the morphometrics. We can use the Davies-Boulding score to identify the best cluster - the one with the lowest score. 

In [ ]:
cgram = clustering(percentiles_joined, num_clusters=30)
clustering_scores_plot(cgram, num_clusters=30)

In [ ]:
merged["cluster"] = cgram.labels[best_cluster_number(cgram, num_clusters=30)].values
buildings["cluster"] = merged["cluster"]

buildings_plot = buildings.copy()

# Plain plot without pretty labels
# buildings.plot(
#     "cluster", categorical=True, figsize=(8, 8), legend=True
# ).set_axis_off()

# Create ordered labels
n_clusters = buildings_plot["cluster"].nunique()
buildings_plot["cluster_label"] = buildings_plot["cluster"].map(lambda x: f"Cluster {x + 1}")
ordered_labels = [f"Cluster {i + 1}" for i in range(n_clusters)]
buildings_plot["cluster_label"] = pd.Categorical(
    buildings_plot["cluster_label"],
    categories=ordered_labels,
    ordered=True
)

fig, ax = plt.subplots(figsize=(8, 8))
buildings_plot.plot(
    "cluster_label", categorical=True, ax=ax, cmap="tab20c", legend=True,
    legend_kwds={"loc": "lower center", "bbox_to_anchor": (0.75, 0.05), "ncol": 3, "frameon": False}
)
ax.set_axis_off()
plt.tight_layout()
plt.show()

## Run the classification with building height
The [Global Building Atlas](https://mediatum.ub.tum.de/1782307) makes available building polygons with height information (derived from satellite imagery) for the world. We can get the data from a single Parquet file for Auckland and Lower North Island from [Source Co-operative](https://source.coop/tge-labs/globalbuildingatlas-lod1). If you're interested in learning more about the data check out this [impressive writeup by Mark Litwintschik](https://tech.marksblogg.com/building-footprints-gba.html). 

In [ ]:
buildings_h, \
streets_h, \
tessellation_h, \
blocks_h, \
merged_h, \
percentiles_joined_h = generate_morphometrics(buildings_raw, streets_raw, include_height=True)

In [ ]:
cgram_h = clustering(percentiles_joined_h, num_clusters=30)
clustering_scores_plot(cgram_h, num_clusters=30)

In [ ]:
merged_h["cluster"] = cgram_h.labels[best_cluster_number(cgram_h, num_clusters=30)].values
buildings_h["cluster"] = merged_h["cluster"]

# Plain plot without pretty labels
# buildings_h.plot(
#     "cluster", categorical=True, figsize=(8, 8), legend=True
# ).set_axis_off()

buildings_h_plot = buildings_h.copy()
buildings_h_plot["cluster_label"] = buildings_h_plot["cluster"].map(lambda x: f"Cluster {x + 1}")

fig, ax = plt.subplots(figsize=(8, 8))
buildings_h_plot.plot(
    "cluster_label", categorical=True, ax=ax, legend=True,
    legend_kwds={"loc": "lower center", "bbox_to_anchor": (0.7, 0.05), "ncol": 3, "frameon": False}
)
ax.set_axis_off()
plt.tight_layout()
plt.show()

## Let's look at how many buildings per cluster
About half of the buildings in the CBD are covered by Clusters 4 and 5. And ~70% are covered by just 4 clusters. Cluster 3 is barely present. Perhaps a spurious building?

In [ ]:
cluster_h_proportion = buildings_h_plot.groupby('cluster_label').count()['cluster'].reset_index()

cluster_h_proportion['proportion'] = cluster_h_proportion['cluster'] / buildings_h_plot.shape[0]

cluster_h_proportion['percentage'] = cluster_h_proportion['proportion'].apply(lambda x: f"{x:.0%}")

cluster_h_proportion

Two clusters again count for around half of the CBD buildings but height is clearly an important classification variable as the remaining half of the buildings need another 12 clusters. 

In [ ]:
cluster_proportion = buildings_plot.groupby('cluster_label').count()['cluster'].reset_index()

cluster_proportion['proportion'] = cluster_proportion['cluster'] / buildings_plot.shape[0]

cluster_proportion['percentage'] = cluster_proportion['proportion'].apply(lambda x: f"{x:.0%}")

cluster_proportion